# LangChain Object Loading Reference

Developer-facing statements defined in `langchain_core.load.load`.

# `DEFAULT_NAMESPACES`

Default trusted root namespaces accepted during deserialization.

```python
DEFAULT_NAMESPACES = [
    "langchain",
    "langchain_core",
    "langchain_community",
    "langchain_anthropic",
    "langchain_groq",
    "langchain_google_genai",
    "langchain_aws",
    "langchain_openai",
    "langchain_google_vertexai",
    "langchain_mistralai",
    "langchain_fireworks",
    "langchain_xai",
    "langchain_sambanova",
    "langchain_perplexity",
]
```

---

# `DISALLOW_LOAD_FROM_PATH`

Namespaces that may be deserialized only through an explicit serialization mapping, not directly from a module path.

```python
DISALLOW_LOAD_FROM_PATH = [
    "langchain_community",
    "langchain",
]
```

---

# `ALL_SERIALIZABLE_MAPPINGS`

Combined current and legacy serialization-path mappings used by the loader.

```python
ALL_SERIALIZABLE_MAPPINGS = {
    **SERIALIZABLE_MAPPING,
    **OLD_CORE_NAMESPACES_MAPPING,
    **_OG_SERIALIZABLE_MAPPING,
    **_JS_SERIALIZABLE_MAPPING,
}
```

---

# `default_init_validator`

Default constructor-argument validator used by `load()`, `loads()`, and `Reviver`.

```python
default_init_validator(
    class_path: tuple[str, ...], # Serialized class path being instantiated
    kwargs: dict[str, Any], # Keyword arguments intended for the constructor
) -> None
```

Raises `ValueError` when `kwargs["template_format"]` is `"jinja2"`. A custom validator may be supplied to replace this behavior.

---

# `AllowedObject`

Type alias for a serializable class accepted by the `allowed_objects` parameter.

```python
AllowedObject = type[Serializable]
```

The value must be a `Serializable` subclass itself, not an instance.

---

# `InitValidator`

Type alias for a callable that validates constructor arguments before deserialization instantiates a class.

```python
InitValidator = Callable[[tuple[str, ...], dict[str, Any]], None]
```

The callable receives the serialized class path and constructor keyword arguments and should raise an exception to reject deserialization.

---

# `Reviver`

Callable JSON-object reviver that reconstructs allowlisted LangChain objects from serialized dictionaries.

## Constructor

```python
Reviver(
    allowed_objects: Iterable[AllowedObject] | Literal["all", "core", "messages"] | None = None, # Classes or predefined allowlist mode
    secrets_map: dict[str, str] | None = None, # Explicit serialized-secret values
    valid_namespaces: list[str] | None = None, # Additional trusted root namespaces
    secrets_from_env: bool = False, # Whether unresolved secrets may be read from environment variables
    additional_import_mappings: dict[tuple[str, ...], tuple[str, ...]] | None = None, # Additional or replacement class-path mappings
    *,
    ignore_unserializable_fields: bool = False, # Return None instead of failing for not-implemented records
    init_validator: InitValidator | None = default_init_validator, # Validator called before class import and construction
) -> None
```

When `allowed_objects` is omitted, the constructor emits a pending deprecation warning marked since version `1.3.3` and currently uses `"core"`.

An explicit iterable must contain `Serializable` subclasses; otherwise construction raises `TypeError`. The predefined modes permit all mapped classes, mapped `langchain_core` classes, or modern message classes respectively.

## Methods

### `__call__`

Revives one parsed JSON dictionary.

```python
__call__(
    self,
    value: dict[str, Any], # Parsed dictionary to inspect and revive
) -> Any # Revived value or the unchanged dictionary
```

## Behaviour

Secret records are resolved from `secrets_map`, then optionally from the environment, and otherwise become `None`.

A `not_implemented` record becomes `None` when `ignore_unserializable_fields=True`; otherwise it raises `NotImplementedError`.

Constructor records must pass the configured class-path allowlist and namespace checks. The validator runs before import, and the resolved class must subclass `Serializable` before it is instantiated with the serialized `kwargs`.

Raises `ValueError` for disallowed class paths, invalid namespaces, unsupported direct-path loading, or validator rejection.

Deserialization instantiates Python classes and can execute their constructors and validators. `"core"` and `"all"` should not be used with untrusted manifests; use `"messages"` or an explicit restrictive class list.

---

# `loads`

Revives LangChain objects from a JSON string.

```python
@beta()
loads(
    text: str, # JSON string to parse and revive
    *,
    allowed_objects: Iterable[AllowedObject] | Literal["all", "core", "messages"] | None = None, # Classes or predefined allowlist mode
    secrets_map: dict[str, str] | None = None, # Explicit serialized-secret values
    valid_namespaces: list[str] | None = None, # Additional trusted root namespaces
    secrets_from_env: bool = False, # Whether unresolved secrets may be read from environment variables
    additional_import_mappings: dict[tuple[str, ...], tuple[str, ...]] | None = None, # Additional or replacement class-path mappings
    ignore_unserializable_fields: bool = False, # Replace not-implemented records with None
    init_validator: InitValidator | None = default_init_validator, # Validator run before object construction
) -> Any # Revived LangChain object structure
```

Equivalent to parsing `text` with `json.loads()` and passing the parsed value to `load()` with the same options.

When `allowed_objects` is omitted, the function emits a pending deprecation warning marked since version `1.3.3` and currently uses `"core"`.

Raises `ValueError` when a serialized class path is not permitted. Exceptions raised by the configured validator or `Reviver` are propagated.

Treat serialized manifests as executable configuration. For untrusted data, use `allowed_objects="messages"` or an explicit restrictive list and keep `secrets_from_env=False`.

---

# `load`

Revives LangChain objects from an already parsed JSON-compatible value.

```python
@beta()
load(
    obj: Any, # Parsed JSON-compatible value to revive
    *,
    allowed_objects: Iterable[AllowedObject] | Literal["all", "core", "messages"] | None = None, # Classes or predefined allowlist mode
    secrets_map: dict[str, str] | None = None, # Explicit serialized-secret values
    valid_namespaces: list[str] | None = None, # Additional trusted root namespaces
    secrets_from_env: bool = False, # Whether unresolved secrets may be read from environment variables
    additional_import_mappings: dict[tuple[str, ...], tuple[str, ...]] | None = None, # Additional or replacement class-path mappings
    ignore_unserializable_fields: bool = False, # Replace not-implemented records with None
    init_validator: InitValidator | None = default_init_validator, # Validator run before object construction
) -> Any # Recursively revived value
```

When `allowed_objects` is omitted, the function emits a pending deprecation warning marked since version `1.3.3` and currently uses `"core"`.

The function recursively processes dictionaries and lists. Escaped dictionaries are unwrapped and returned as plain user data without being interpreted as LangChain constructor records.

Raises `ValueError` when a serialized class path is not permitted. Exceptions raised by the configured validator or `Reviver` are propagated.

Treat serialized manifests as executable configuration. For untrusted data, use `allowed_objects="messages"` or an explicit restrictive list and keep `secrets_from_env=False`.

In [ ]:
from langchain_core.load import dumpd, dumps, load, loads # Import serialization and loading functions
from langchain_core.messages import AIMessage # Import a LangChain message class

message = AIMessage(content="Hello, Saad!") # Create a sample LangChain object

dictionary_data = dumpd(message) # Convert the message into a dictionary
json_data = dumps(message, pretty=True) # Convert the message into a JSON string

loaded_from_dictionary = load(
    dictionary_data,
    allowed_objects=[AIMessage],
) # Restore the message from the dictionary

loaded_from_json = loads(
    json_data,
    allowed_objects=[AIMessage],
) # Restore the message from the JSON string

print("Serialized dictionary:", dictionary_data) # Display the dictionary
print("Serialized JSON:", json_data) # Display the JSON string
print("Loaded using load():", loaded_from_dictionary.content) # Display restored content
print("Loaded using loads():", loaded_from_json.content) # Display restored content